# 03 - Asset pricing tests

Notebook audit entry point for the econometrics team's Steps 1-6. It reads the committed test inputs under `data/`, calls reusable implementations from `src/`, and displays the saved step outputs. Step 7 is pending Vietnam returns and Vietnam factors.

Run from the repository root after installing `requirements.lock.txt`. The Python scripts remain the canonical full workflow, including vintage-sensitive comparisons.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the repository root or its notebooks/ directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / 'data' / 'test' / 'kenneth_french'
OUTPUTS = ROOT / 'outputs'
portfolio_returns = pd.read_csv(DATA / '25_portfolios_size_bm.csv', index_col=0, parse_dates=True)
ff3 = pd.read_csv(DATA / 'ff3_factors_monthly.csv', index_col=0, parse_dates=True)
ff5 = pd.read_csv(DATA / 'ff5_factors_monthly.csv', index_col=0, parse_dates=True)
print(f'Root: {ROOT}')
print(f'LHS: {portfolio_returns.shape}; FF3: {ff3.shape}; FF5: {ff5.shape}')

## Step 1 - 25 dependent portfolios

Columns are the 5 x 5 Size-B/M portfolio grid. Rows of the conceptual grid are size groups; columns are book-to-market groups. `SMALL LoBM` is the single LHS used for the single-portfolio model comparison.

In [ ]:
print(f'{portfolio_returns.index.min():%Y-%m} through {portfolio_returns.index.max():%Y-%m}; {len(portfolio_returns)} months')
display(pd.DataFrame({'portfolio_column': portfolio_returns.columns}))
display(portfolio_returns[['SMALL LoBM', 'ME1 BM2', 'ME1 BM3', 'ME1 BM4', 'SMALL HiBM']].head())

## Steps 2-3 - OLS and model comparison

The shared function subtracts RF from the raw portfolio return and returns a statsmodels result. The stored comparison table is generated by `scripts/compare_factor_models.py`.

In [ ]:
from src.models.regressions import run_factor_regression

y = portfolio_returns['SMALL LoBM']
models = {
    'CAPM': run_factor_regression(y, ff3, ['Mkt-RF']),
    'FF3': run_factor_regression(y, ff3, ['Mkt-RF', 'SMB', 'HML']),
    'FF5': run_factor_regression(y, ff5, ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']),
}
display(pd.DataFrame([{'model': name, 'alpha': fit.params['const'], 't_alpha': fit.tvalues['const'], 'R2': fit.rsquared, 'nobs': int(fit.nobs)} for name, fit in models.items()]))
models['FF5'].summary()

In [ ]:
display(pd.read_csv(OUTPUTS / 'step3_model_comparison.csv'))

## Step 4 - Published-result comparison

These saved tables compare the current test sample with the cited Fama-French (2015) Table 5 and Table 7 values. Historical factor archives are under `data/reference/`; the local 25-portfolio LHS vintage remains unverified, so this is an approximate comparison.

In [ ]:
display(pd.read_csv(OUTPUTS / 'step4_table5_panel_a_comparison.csv'))
display(pd.read_csv(OUTPUTS / 'step4_table7_comparison.csv'))
print((OUTPUTS / 'step4_audit.md').resolve())

## Step 5 - Joint GRS test

The canonical implementation in `src/models/grs_test.py` tests the 25 alphas jointly. The saved table includes current factor files and historical factor-vintage sensitivity rows.

In [ ]:
from src.models.grs_test import grs_test

grs_current = {
    'FF3': grs_test(portfolio_returns, ff3, ['Mkt-RF', 'SMB', 'HML']),
    'FF5': grs_test(portfolio_returns, ff5, ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']),
}
display(pd.DataFrame([{'model': name, 'GRS': result.statistic, 'p_value': result.p_value, 'T': result.observations, 'N': result.n_portfolios, 'K': result.n_factors} for name, result in grs_current.items()]))
display(pd.read_csv(OUTPUTS / 'step5_grs_results.csv'))

## Step 6 - HAC and VIF

HAC changes the covariance estimate used for inference; it does not change OLS coefficients. VIF reports collinearity among the selected factors. Saved outputs include the primary and sensitivity HAC bandwidths.

In [ ]:
from src.models.diagnostics import calculate_vif, run_hac_regression

factor_specs = {
    'FF3': (ff3, ['Mkt-RF', 'SMB', 'HML']),
    'FF5': (ff5, ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']),
}
vif_tables = [calculate_vif(frame, factors).assign(model=name) for name, (frame, factors) in factor_specs.items()]
display(pd.concat(vif_tables, ignore_index=True))
hac_models = {name: run_hac_regression(y, frame, factors, maxlags=12) for name, (frame, factors) in factor_specs.items()}
display(pd.DataFrame([{'model': name, 'alpha': float(result.hac.params[0]), 'hac_se_alpha': float(result.hac.bse[0]), 'hac_t_alpha': float(result.hac.tvalues[0]), 'maxlags': result.maxlags} for name, result in hac_models.items()]))

display(pd.read_csv(OUTPUTS / 'step6_hac_results.csv'))
display(pd.read_csv(OUTPUTS / 'step6_vif.csv'))
print((OUTPUTS / 'step6_audit.md').resolve())

## Step 7 - Vietnam data gate

This notebook uses U.S. reference/test data only. Do not interpret these results as Vietnam estimates. Run Step 7 only after the Vietnam LHS returns and the Factor Team's Vietnam factor series have been supplied and aligned.